In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import TSMixerModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.timeseries_generation import datetime_attribute_timeseries

# 1. 데이터 로드
df = pd.read_csv('../data/processed/smart_meter_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# 2. Darts 시계열 객체로 변환 (유량과 수압을 함께 넣음)
series = TimeSeries.from_dataframe(df, 
                                   time_col='Timestamp', 
                                   value_cols=['Flow_Instant', 'Pressure'], 
                                   freq='10T') # 10분 단위

# 3. 데이터 전처리 (스케일링: 0~1 사이 값으로 변환)
scaler = Scaler()
series_scaled = scaler.fit_transform(series)

# 4. 학습/검증 데이터 분리 (마지막 30일을 테스트용으로)
train, val = series_scaled.split_before(pd.Timestamp('2025-03-01'))

# 5. TSMixer 모델 설정
# input_chunk_length: 과거 14일 (14 * 24 * 6 = 2016)
# output_chunk_length: 미래 7일 (7 * 24 * 6 = 1008)
model = TSMixerModel(
    input_chunk_length=2016, 
    output_chunk_length=1008,
    n_epochs=20,
    batch_size=32,
    hidden_size=64,
    ff_size=64,
    num_blocks=2,
    random_state=42
)

# 6. 모델 학습
print("🚀 모델 학습 시작...")
model.fit(train)

# 7. 예측 수행 (검증 데이터의 시작점부터 7일치 예측)
prediction = model.predict(n=1008, series=train)

# 8. 역스케일링 (원래 수치로 복원)
prediction_rescaled = scaler.inverse_transform(prediction)
val_rescaled = scaler.inverse_transform(val)

# 9. 결과 시각화
plt.figure(figsize=(15, 10))

# 유량(Flow) 결과
plt.subplot(2, 1, 1)
val_rescaled['Flow_Instant'][:1008].plot(label='Actual Flow', color='gray')
prediction_rescaled['Flow_Instant'].plot(label='Predicted Flow', color='blue')
plt.title('7-Day Flow Rate Prediction (TSMixer)')
plt.legend()

# 수압(Pressure) 결과
plt.subplot(2, 1, 2)
val_rescaled['Pressure'][:1008].plot(label='Actual Pressure', color='gray')
prediction_rescaled['Pressure'].plot(label='Predicted Pressure', color='red')
plt.title('7-Day Pressure Prediction (TSMixer)')
plt.legend()

plt.tight_layout()
plt.show()
